In [1]:
from spin_lattices import TriangleLattice, SquareLattice, KagomeLattice,  SpinLattice
from heisenberg_hamiltonians import HeisenbergJ1J2
from fast_boolean_analysis import FourierSeries, fourier_expand, keep_largest_n
from lattice_boolean_analysis import LBFFromSpinSystem
from pathlib import Path
import numpy as np
from nn_xors_2023_07_18 import make_dataset, train, MLPBinaryClassifier
from loguru import logger
import torch
from torch.utils.data import random_split, DataLoader
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torch import nn

2023-07-20 20:11:24.011 | DEBUG    | lattice_symmetries:__init__:50 - Initializing Haskell runtime...
2023-07-20 20:11:24.016 | DEBUG    | lattice_symmetries:__init__:52 - Initializing Chapel runtime...
2023-07-20 20:11:24.066 | DEBUG    | lattice_symmetries:__init__:54 - Setting Python exception handler...
set_python_exception_handler ...


In [2]:
lattice = KagomeLattice(2, 4)
system = HeisenbergJ1J2(lattice, J1=1, J2=0.8, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)

2023-07-20 20:11:28.838 | DEBUG    | heisenberg_hamiltonians:__init__:435 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-07-20 20:11:28.840 | DEBUG    | heisenberg_hamiltonians:__init__:456 - number_spins=24
2023-07-20 20:11:28.848 | DEBUG    | heisenberg_hamiltonians:__init__:466 - Symmetry group contains 16 elements


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-20 20:11:28.917 | DEBUG    | heisenberg_hamiltonians:__init__:475 - Hilbert space dimension is 85662
2023-07-20 20:11:28.927 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:64 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.8-True-1-1.pickle
2023-07-20 20:11:28.930 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:111 - Ground state energy is -40.5183420067


(array([-40.51834201]),
 array([[ 8.36163038e-08],
        [ 1.13414544e-07],
        [ 1.79331566e-08],
        ...,
        [-8.66655973e-03],
        [-1.17282633e-02],
        [ 3.99580256e-03]]))

In [3]:
signal = LBFFromSpinSystem(system)
series = fourier_expand(signal)

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-20 20:11:29.104 | DEBUG    | fast_boolean_analysis:fourier_expand:295 - Finding signal
2023-07-20 20:11:29.105 | DEBUG    | heisenberg_hamiltonians:get_ground_state_in_full_basis:196 - Finding coeffs
2023-07-20 20:11:29.106 | DEBUG    | heisenberg_hamiltonians:get_ground_state_in_canonical_basis:218 - Finding basis_state_info
2023-07-20 20:11:29.525 | DEBUG    | heisenberg_hamiltonians:get_ground_state_in_canonical_basis:221 - Finding corresp_repr_indices
2023-07-20 20:11:29.555 | DEBUG    | heisenberg_hamiltonians:get_ground_state_in_canonical_basis:227 - Finding coeffs
/vol/tcm10/ischurov/frustrations-eda/heisenberg_hamiltonians.py:199: ComplexWarning: Casting complex values to real discards the imaginary part
  coeffs[self.canonical_basis.states] = self.get_ground_state_in_canonical_basis()
2023-07-20 20:11:29.844 | DEBUG    | fast_boolean_analysis:fourier_expand:297 - Doing 


In [4]:
series.truncate(keep_largest_n(5))(system.canonical_basis.states[:10])

array([ 0.00583768, -0.02546668, -0.00583768, -0.00583768,  0.00583768,
        0.00583768, -0.03714204, -0.00583768, -0.00583768,  0.00583768])

In [5]:
def train(
    net,
    criterion,
    optimizer,
    train_loader,
    test_dataset,
    n_epochs,
    writer: SummaryWriter,
    break_on_loss: None | float = None,
):
    for epoch in range(n_epochs):
        hits = 0
        for i, (x, y) in enumerate(train_loader):
            optimizer.zero_grad()
            yhat = net(x)
            loss = criterion(yhat, y)
            loss.backward()
            optimizer.step()
            hits += (yhat.argmax(dim=1) == y).float().sum()
        train_accuracy = hits / len(train_loader.dataset)

        writer.add_scalar("train/loss", loss.item(), epoch)
        writer.add_scalar("train/accuracy", train_accuracy, epoch)
        with torch.no_grad():
            x, y = test_dataset[:]
            yhat = net(x)
            # find accuracy
            test_accuracy = (yhat.argmax(dim=1) == y).float().mean()
            writer.add_scalar("test/accuracy", test_accuracy, epoch)
            logger.debug(
                f"Epoch\t{epoch}\tloss\t{loss.item():.4f}"
                f"\taccuracy\t{train_accuracy:.4f}\tAccuracy (test)\t{test_accuracy:.4f}"
            )
        if break_on_loss is not None and loss < break_on_loss:
            break
    return {
        "loss": loss.item(),
        "train_accuracy": train_accuracy.item(),
        "test_accuracy": test_accuracy.item(),
        "epoch": epoch,
    }

In [9]:
lattice = SquareLattice(2 * 3, 4)
eps_train = 0.01
eps_test = 0.1
batch_size = 64
n_hidden = 64

system = HeisenbergJ1J2(lattice, J1=1, J2=0.5, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)
signal = LBFFromSpinSystem(system)
series = fourier_expand(signal)

n_spins = system.number_spins
results = []
for keep in [None, 5, 10, 20, 50]:
    writer = SummaryWriter(
        log_dir=(
            f"experiments/2023_07_20/{datetime.now().strftime('%H_%M_%S')}"
            #            f"_{n_xors=}_{xor_hamming_weight=}_{distance=}_{run=}"
        )
    )
    all_states = system.canonical_basis.states
    sample_states = np.random.choice(
        all_states,
        size=int(len(all_states) * (eps_train + eps_test)),
        replace=False,
    )

    if keep is not None:
        truncated_series = series.truncate(keep_largest_n(keep))
    else:
        truncated_series = system.get_ground_state_coeffs

    dataset = make_dataset(truncated_series, sample_states, n_spins)
    train_dataset, test_dataset = random_split(
        dataset, [eps_train / (eps_train + eps_test), eps_test / (eps_train + eps_test)]
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

    net = MLPBinaryClassifier(n_spins, n_hidden)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

    output = train(
        net,
        criterion,
        optimizer,
        train_loader,
        test_dataset=test_dataset,
        n_epochs=500,
        writer=writer,
        break_on_loss=1e-04,
    )
    results.append(output)
    logger.debug(f"Finished training, {output=}")

2023-07-20 20:24:46.370 | DEBUG    | heisenberg_hamiltonians:__init__:435 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-07-20 20:24:46.371 | DEBUG    | heisenberg_hamiltonians:__init__:456 - number_spins=24


2023-07-20 20:24:46.405 | DEBUG    | heisenberg_hamiltonians:__init__:466 - Symmetry group contains 96 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-20 20:24:46.521 | DEBUG    | heisenberg_hamiltonians:__init__:475 - Hilbert space dimension is 15578
2023-07-20 20:24:46.545 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:64 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-SquareLattice6x4-1.0-0.5-True-1-1.pickle
2023-07-20 20:24:46.547 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:111 - Ground state energy is -50.1623937953
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-20 20:24:46.608 | DEBUG    | fast_boolean_analysis:fourier_expand:295 - Finding signal
2023-07-20 20:24:46.609 | DEBUG    | heisenberg_hamiltonians:get_ground_state_in_full_basis:196 - Finding coeffs
2023-07-20 20:24:46.611 | DEBUG    | heisenberg_hamiltonians:get_ground_state_in_canonical_basis:218 - Finding basis

RuntimeError: [enforce fail at alloc_cpu.cpp:75] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 69226240 bytes. Error code 12 (Cannot allocate memory)

In [31]:
(net(X).argmax(dim=1) == Y).float().mean()

tensor(0.6094)